# BAN 612 Week 5 — Data Wrangling Notebook

**Business context:** A B2B SaaS company wants to combine revenue, product usage, support tickets, and customer master data into a customer-month table.

This notebook is annotated for in-class demonstration. It emphasizes **wrangling decisions**: granularity, reshaping, feature construction, method chaining, groupby aggregation, and joining transformed tables.

Dataset files are included separately in `Week5_SaaS_Data` folder.

## Learning goals

By the end of this notebook, students should be able to:

1. Convert transactional data into analysis-ready feature tables.
2. Use method chaining with `.assign()`, `.pipe()`, `.groupby()`, and named aggregations.
3. Reshape wide data to long format and long data to summary tables.
4. Align tables at the same unit of analysis before merging.
5. Explain how wrangling choices affect the business question.

In [1]:
# Core libraries for this notebook.
# pandas is used for data wrangling; numpy is used for numerical helpers; pathlib helps with file paths.
import pandas as pd
import numpy as np
from pathlib import Path


# Display settings make tables easier to read in class.
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)

## 1. Load the files

The focus this week is not basic loading or validation. We still load the files, but the main task is to **transform multiple source tables into a useful analytic table**.

In [5]:
# Set the data directory and load the data (all 6 files)
data_dir = Path('Week5_SaaS_Data')

customers = pd.read_csv(data_dir / 'customers.csv')
subscriptions = pd.read_csv(data_dir / 'subscriptions.csv')
invoices = pd.read_csv(data_dir/ 'invoices.csv')
usage = pd.read_csv(data_dir/ 'weekly_usage.csv')
tickets = pd.read_csv(data_dir / 'support_tickets.csv')
usage_wide = pd.read_csv(data_dir / 'usage_wide_demo.csv')

# Quick row counts. This is not the main analysis, but it gives us orientation.
for name, df in {
    'customers': customers,
    'subscriptions': subscriptions,
    'invoices': invoices,
    'usage': usage,
    'tickets': tickets,
    'usage_wide': usage_wide
}.items():
    print(f'{name:15s}', df.shape)

customers       (140, 7)
subscriptions   (158, 8)
invoices        (1142, 8)
usage           (5928, 7)
tickets         (300, 7)
usage_wide      (134, 7)


In [6]:
customers.head()

,customer_id,company_name,signup_date,industry,company_size,region,acquisition_channel
0,C1000,Evergreen Analytics 0,2024-07-30,Financial Services,51-200,West,Outbound
1,C1001,Falcon Group 1,2024-12-01,Professional Services,201-1000,South,Referral
2,C1002,Civic Partners 2,2024-05-19,Healthcare,51-200,South,Outbound
3,C1003,Falcon Systems 3,2024-02-10,Retail,1001+,West,Webinar
4,C1004,Bright Analytics 4,2024-02-08,Professional Services,201-1000,Northeast,Partner


In [7]:
subscriptions.head()

,subscription_id,customer_id,plan_name,start_date,end_date,billing_cycle,mrr,subscription_status
0,S5000,C1000,Growth,2024-07-30,NaN,Annual,245.00,Active
1,S5001,C1001,Growth,2024-12-10,NaN,Monthly,207.23,Active
2,S5002,C1002,Starter,2024-06-01,2025-10-01,Monthly,89.78,Churned
3,S5003,C1003,Starter,2024-02-12,2025-03-12,Annual,100.13,Churned
4,S5004,C1004,Starter,2024-02-15,2025-01-14,Monthly,110.84,Upgraded


In [8]:
invoices.head()

,invoice_id,customer_id,invoice_date,amount_due,amount_paid,paid_date,discount_code,refund_amount
0,I10000,C1000,2025-01-02,2646.00,2646.00,2025-01-04,NEW10,0.00
1,I10001,C1001,2025-01-02,207.23,207.23,2025-01-07,NONE,0.00
2,I10002,C1001,2025-02-03,165.78,165.78,2025-02-03,SAVE20,0.00
3,I10003,C1001,2025-03-01,207.23,0.00,NaN,NONE,103.62
4,I10004,C1001,2025-04-02,186.51,93.26,2025-04-02,NEW10,93.25


In [9]:
usage.head()

,customer_id,week_start,active_users,logins,reports_created,api_calls,storage_gb
0,C1000,2025-01-06,8,18,11,655,9.35
1,C1000,2025-01-13,9,46,14,706,8.72
2,C1000,2025-01-20,3,7,6,220,2.07
3,C1000,2025-01-27,12,57,16,949,11.46
4,C1000,2025-02-03,3,15,5,265,0.18


In [10]:
tickets.head()

,ticket_id,customer_id,opened_at,closed_at,ticket_category,priority,csat_score
0,T8000,C1000,2025-08-22 09:00:00,2025-08-22 17:00:00,API,High,3.0
1,T8001,C1000,2025-03-09 11:00:00,2025-03-14 11:00:00,Billing,Low,4.0
2,T8002,C1001,2025-07-31 14:00:00,2025-08-01 02:00:00,Login,Medium,3.0
3,T8003,C1001,2025-03-17 14:00:00,2025-03-18 02:00:00,Billing,Low,4.0
4,T8004,C1001,2025-05-31 16:00:00,2025-06-01 16:00:00,Billing,Medium,2.0


In [11]:
usage_wide.head()

,customer_id,logins_01,logins_02,logins_03,logins_04,logins_05,logins_06
0,C1000,128.0,107.0,133.0,107.0,106.0,122.0
1,C1001,103.0,71.0,163.0,77.0,100.0,123.0
2,C1002,27.0,34.0,79.0,34.0,36.0,56.0
3,C1003,68.0,42.0,27.0,NaN,NaN,NaN
4,C1004,59.0,122.0,124.0,111.0,114.0,143.0


## 2. Start with the business question and unit of analysis

The SaaS VP asks:

> Which customers show signs of weak engagement or revenue risk by month?

That implies the target table should be at the **customer-month** level.

Before merging, every source table must be transformed to the same grain: one row per `customer_id` per month.

In [12]:
#Identify the time period in the data
subscriptions["start_date"] = pd.to_datetime(
    subscriptions["start_date"],
    errors="coerce"
)

subscriptions["end_date"] = pd.to_datetime(
    subscriptions["end_date"],
    errors="coerce"
)

print("Earliest subscription start:",
      subscriptions["start_date"].min())

print("Latest subscription end:",
      subscriptions["end_date"].max())

Earliest subscription start: 2024-01-07 00:00:00
Latest subscription end: 2025-11-24 00:00:00


In [13]:
#Try to get the time period from other sources.
usage["week_start"] = pd.to_datetime(
    usage["week_start"],
    errors="coerce"
)

print("Earliest Week in the usage data:",
      usage["week_start"].min())

print("Latest week in the usage data:",
      usage["week_start"].max())

Earliest Week in the usage data: 2025-01-06 00:00:00
Latest week in the usage data: 2025-12-29 00:00:00


In [21]:
# We will use this variable repeatedly.
# A month stored as Period[M] is useful for grouping by calendar month without worrying about day/time.
analysis_start = pd.Period('2025-01', freq='M')
analysis_end = pd.Period('2025-12', freq='M')

## 3. Method chaining: transform without losing the story

Method chaining lets us read a wrangling pipeline from top to bottom.

Common pattern:

```python
(df
 .assign(new_column = ...)
 .query(...)
 .groupby(...)
 .agg(...)
 .reset_index()
)
```

This is useful because the code itself documents the transformation sequence.

In [15]:
#Identify customer groups through customer_size variable
customers["company_size"].value_counts(dropna=False)

company_size
1-50        47
51-200      42
201-1000    34
1001+       17
Name: count, dtype: int64

In [16]:
# Prepare customer master data.
# Note that we do not need to do heavy cleaning here; we create fields useful for later joins and analysis.
customers_wr = (
    customers
    .assign(
        signup_date = lambda d: pd.to_datetime(d['signup_date'], errors='coerce'),
        signup_month = lambda d: d['signup_date'].dt.to_period('M'),
        # Simplify company size into a rough segment for analysis.
        size_segment = lambda d: np.where(d['company_size'].isin(['1-50', '51-200']), 'SMB', 'Mid-Market / Enterprise')
    )
)

customers_wr.head()

,customer_id,company_name,signup_date,industry,company_size,region,acquisition_channel,signup_month,size_segment
0,C1000,Evergreen Analytics 0,2024-07-30,Financial Services,51-200,West,Outbound,2024-07,SMB
1,C1001,Falcon Group 1,2024-12-01,Professional Services,201-1000,South,Referral,2024-12,Mid-Market / Enterprise
2,C1002,Civic Partners 2,2024-05-19,Healthcare,51-200,South,Outbound,2024-05,SMB
3,C1003,Falcon Systems 3,2024-02-10,Retail,1001+,West,Webinar,2024-02,Mid-Market / Enterprise
4,C1004,Bright Analytics 4,2024-02-08,Professional Services,201-1000,Northeast,Partner,2024-02,Mid-Market / Enterprise


## 4. Wrangle invoices into customer-month revenue features

`invoices.csv` is invoice-level data. The analytic table requires customer-month data.

We will create:

- `invoice_count`
- `gross_billed`
- `net_collected`
- `refund_total`
- `pct_paid`
- `avg_days_to_pay`

The important idea is that we are not merely cleaning columns; we are **constructing features**.

In [19]:
# Create invoice-level features before aggregating.
# .assign() adds or overwrites columns while preserving a readable pipeline.
invoices_wr = (
    invoices
    .assign(
        invoice_date = lambda d: pd.to_datetime(d['invoice_date'], errors='coerce'),
        #do for paid_date
        paid_date = lambda d: pd.to_datetime(d['paid_date'], errors='coerce'),
        invoice_month = lambda d: d['invoice_date'].dt.to_period('M'),
        was_paid = lambda d: d['amount_paid'] > 0,
        #Calculate net_collected
        net_collected = lambda d: (d['amount_paid'] - d['refund_amount']),
        # days_to_pay is missing when the invoice was not paid. That is expected, not necessarily an error.
        days_to_pay = lambda d: (d['paid_date'] - d['invoice_date']).dt.days
    )
)

invoices_wr[['invoice_id','customer_id','invoice_month','amount_due','amount_paid','refund_amount','net_collected','was_paid','days_to_pay']].head()

,invoice_id,customer_id,invoice_month,amount_due,amount_paid,refund_amount,net_collected,was_paid,days_to_pay
0,I10000,C1000,2025-01,2646.00,2646.00,0.00,2646.00,True,2.0
1,I10001,C1001,2025-01,207.23,207.23,0.00,207.23,True,5.0
2,I10002,C1001,2025-02,165.78,165.78,0.00,165.78,True,0.0
3,I10003,C1001,2025-03,207.23,0.00,103.62,-103.62,False,NaN
4,I10004,C1001,2025-04,186.51,93.26,93.25,0.01,True,0.0


In [22]:
revenue_monthly = (
    invoices_wr
    .query('@analysis_start <= invoice_month <= @analysis_end')
    .groupby(['customer_id', 'invoice_month'], as_index=False)
    
)

revenue_monthly.head()

,invoice_id,customer_id,invoice_date,amount_due,amount_paid,paid_date,discount_code,refund_amount,invoice_month,was_paid,net_collected,days_to_pay
0,I10000,C1000,2025-01-02,2646.00,2646.00,2025-01-04,NEW10,0.00,2025-01,True,2646.00,2.0
1,I10001,C1001,2025-01-02,207.23,207.23,2025-01-07,NONE,0.00,2025-01,True,207.23,5.0
2,I10002,C1001,2025-02-03,165.78,165.78,2025-02-03,SAVE20,0.00,2025-02,True,165.78,0.0
3,I10003,C1001,2025-03-01,207.23,0.00,NaT,NONE,103.62,2025-03,False,-103.62,NaN
4,I10004,C1001,2025-04-02,186.51,93.26,2025-04-02,NEW10,93.25,2025-04,True,0.01,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1137,I11137,C1139,2025-08-03,605.94,605.94,2025-08-18,NONE,0.00,2025-08,True,605.94,15.0
1138,I11138,C1139,2025-09-03,605.94,605.94,2025-09-05,NONE,0.00,2025-09,True,605.94,2.0
1139,I11139,C1139,2025-10-03,484.75,484.75,2025-10-03,SAVE20,0.00,2025-10,True,484.75,0.0
1140,I11140,C1139,2025-11-01,484.75,484.75,2025-11-01,SAVE20,0.00,2025-11,True,484.75,0.0


In [23]:
# Aggregate invoices to customer-month.
# Named aggregation creates clear output column names.
revenue_monthly = (
    invoices_wr
    .query('@analysis_start <= invoice_month <= @analysis_end')
    .groupby(['customer_id', 'invoice_month'], as_index=False)
    .agg(
        invoice_count = ('invoice_id', 'nunique'),
        gross_billed = ('amount_due', 'sum'),
        net_collected = ('net_collected', 'sum'),
        refund_total = ('refund_amount', 'sum'),
        paid_invoices = ('was_paid', 'sum'),
        avg_days_to_pay = ('days_to_pay', 'mean'),
    )
    .assign(
        pct_paid = lambda d: d['paid_invoices'] / d['invoice_count']
    )
)

revenue_monthly.head()

,customer_id,invoice_month,invoice_count,gross_billed,net_collected,refund_total,paid_invoices,avg_days_to_pay,pct_paid
0,C1000,2025-01,1,2646.00,2646.00,0.00,1,2.0,1.0
1,C1001,2025-01,1,207.23,207.23,0.00,1,5.0,1.0
2,C1001,2025-02,1,165.78,165.78,0.00,1,0.0,1.0
3,C1001,2025-03,1,207.23,-103.62,103.62,0,NaN,0.0
4,C1001,2025-04,1,186.51,0.01,93.25,1,0.0,1.0


## 5. Wrangle weekly usage into customer-month engagement features

`weekly_usage.csv` has one row per customer-week. We will aggregate it to customer-month.

The choice of aggregation depends on the meaning of the variable:

- `logins`, `reports_created`, `api_calls`: usually summed over the month.
- `active_users`: often averaged or maxed.
- `storage_gb`: often maxed or averaged, depending on business meaning.

In [24]:
usage_wr = (
    usage
    .assign(
        week_start = lambda d: pd.to_datetime(d['week_start'], errors='coerce'),
        usage_month = lambda d: d['week_start'].dt.to_period('M')
    )
)

usage_monthly = (
    usage_wr
    .query('@analysis_start <= usage_month <= @analysis_end')
    .groupby(['customer_id', 'usage_month'], as_index=False)
    .agg(
        monthly_logins = ('logins', 'sum'),
        monthly_reports = ('reports_created', 'sum'),
        monthly_api_calls = ('api_calls', 'sum'),
        avg_active_users = ('active_users', 'mean'),
        max_storage_gb = ('storage_gb', 'max'),
    )
    .rename(columns={'usage_month':'month'})
)

usage_monthly.head()

,customer_id,month,monthly_logins,monthly_reports,monthly_api_calls,avg_active_users,max_storage_gb
0,C1000,2025-01,128,47,2530,8.00,11.46
1,C1000,2025-02,107,37,2158,6.75,8.09
2,C1000,2025-03,133,61,2927,7.40,10.49
3,C1000,2025-04,107,46,2320,7.25,7.84
4,C1000,2025-05,106,43,2217,7.00,10.72


## 6. Wrangle support tickets into customer-month service features

Support tickets are event-level records. We will create monthly features such as:

- number of tickets
- number of high-priority tickets
- average resolution hours
- average CSAT score

These are not raw data fields; they are business features built from events.

In [25]:
tickets_wr = (
    tickets
    .assign(
        opened_at = lambda d: pd.to_datetime(d['opened_at'], errors='coerce'),
        closed_at = lambda d: pd.to_datetime(d['closed_at'], errors='coerce'),
        ticket_month = lambda d: d['opened_at'].dt.to_period('M'),
        resolution_hours = lambda d: (d['closed_at'] - d['opened_at']).dt.total_seconds() / 3600,
        high_priority = lambda d: d['priority'].isin(['High','Urgent'])
    )
)

support_monthly = (
    tickets_wr
    .query('@analysis_start <= ticket_month <= @analysis_end')
    .groupby(['customer_id', 'ticket_month'], as_index=False)
    .agg(
        ticket_count = ('ticket_id', 'nunique'),
        high_priority_tickets = ('high_priority', 'sum'),
        avg_resolution_hours = ('resolution_hours', 'mean'),
        avg_csat = ('csat_score', 'mean'),
    )
    .rename(columns={'ticket_month':'month'})
)

support_monthly.head()

,customer_id,month,ticket_count,high_priority_tickets,avg_resolution_hours,avg_csat
0,C1000,2025-03,1,0,120.0,4.0
1,C1000,2025-08,1,1,8.0,3.0
2,C1001,2025-01,1,1,8.0,1.0
3,C1001,2025-03,1,0,12.0,4.0
4,C1001,2025-05,1,0,24.0,2.0


## 7. Reshaping: wide usage table to long format

Wide tables are common in spreadsheets. For analysis, a long format is often easier.

The file `usage_wide_demo.csv` contains columns such as `logins_01`, `logins_02`, etc. We will convert it to:

```text
customer_id | month | logins
```

This is a classic data-wrangling task.

In [26]:
usage_wide.head()

,customer_id,logins_01,logins_02,logins_03,logins_04,logins_05,logins_06
0,C1000,128.0,107.0,133.0,107.0,106.0,122.0
1,C1001,103.0,71.0,163.0,77.0,100.0,123.0
2,C1002,27.0,34.0,79.0,34.0,36.0,56.0
3,C1003,68.0,42.0,27.0,NaN,NaN,NaN
4,C1004,59.0,122.0,124.0,111.0,114.0,143.0


In [27]:
usage_long_demo = (
    usage_wide
    .melt(
        id_vars='customer_id',           # columns that identify the row and should stay fixed
        var_name='month_label',          # name for the old column headers
        value_name='logins'              # name for the values previously spread across columns
    )
    .assign(
        month_number = lambda d: d['month_label'].str.extract(r'(\d{2})$')[0],
        month = lambda d: pd.PeriodIndex('2025-' + d['month_number'], freq='M')
    )
    .drop(columns=['month_label','month_number'])
    .sort_values(['customer_id','month'])
)

usage_long_demo.head(12)

,customer_id,logins,month
0,C1000,128.0,2025-01
134,C1000,107.0,2025-02
268,C1000,133.0,2025-03
402,C1000,107.0,2025-04
536,C1000,106.0,2025-05
670,C1000,122.0,2025-06
1,C1001,103.0,2025-01
135,C1001,71.0,2025-02
269,C1001,163.0,2025-03
403,C1001,77.0,2025-04


## 8. Create a complete customer-month skeleton

If we only merge months that appear in source tables, missing activity can disappear.

A skeleton table makes the intended grain explicit: every customer, every month in 2025.

In [30]:
months = pd.period_range(analysis_start, analysis_end, freq='M')

customer_month_skeleton = (
    customers_wr[['customer_id']]
    .assign(key=1)
    .merge(pd.DataFrame({'month': months, 'key':1}), on='key')
    .drop(columns='key')
)

customer_month_skeleton.head(), customer_month_skeleton.shape

(  customer_id    month
 0       C1000  2025-01
 1       C1000  2025-02
 2       C1000  2025-03
 3       C1000  2025-04
 4       C1000  2025-05,
 (1680, 2))

## 9. Merge transformed monthly feature tables

Now all tables share the same grain: `customer_id` + `month`.

This is the safest point to merge. Joining event-level tables directly would create duplicated rows and inflated totals.

In [31]:
# Rename invoice_month to month before merging.
revenue_features = revenue_monthly.rename(columns={'invoice_month':'month'})

customer_month = (
    customer_month_skeleton
    .merge(revenue_features, on=['customer_id','month'], how='left', validate='one_to_one')
    .merge(usage_monthly, on=['customer_id','month'], how='left', validate='one_to_one')
    .merge(support_monthly, on=['customer_id','month'], how='left', validate='one_to_one')
    .merge(customers_wr[['customer_id','industry','region','acquisition_channel','signup_month','size_segment']],
           on=['customer_id'], how='left', validate='many_to_one')
)

customer_month.head()

,customer_id,month,invoice_count,gross_billed,net_collected,refund_total,paid_invoices,avg_days_to_pay,pct_paid,monthly_logins,monthly_reports,monthly_api_calls,avg_active_users,max_storage_gb,ticket_count,high_priority_tickets,avg_resolution_hours,avg_csat,industry,region,acquisition_channel,signup_month,size_segment
0,C1000,2025-01,1.0,2646.0,2646.0,0.0,1.0,2.0,1.0,128.0,47.0,2530.0,8.00,11.46,NaN,NaN,NaN,NaN,Financial Services,West,Outbound,2024-07,SMB
1,C1000,2025-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,107.0,37.0,2158.0,6.75,8.09,NaN,NaN,NaN,NaN,Financial Services,West,Outbound,2024-07,SMB
2,C1000,2025-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,133.0,61.0,2927.0,7.40,10.49,1.0,0.0,120.0,4.0,Financial Services,West,Outbound,2024-07,SMB
3,C1000,2025-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,107.0,46.0,2320.0,7.25,7.84,NaN,NaN,NaN,NaN,Financial Services,West,Outbound,2024-07,SMB
4,C1000,2025-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106.0,43.0,2217.0,7.00,10.72,NaN,NaN,NaN,NaN,Financial Services,West,Outbound,2024-07,SMB


## 10. Fill missing values only when the business meaning is clear

After the merge, missing values have different meanings:

- Missing `monthly_logins` likely means no recorded usage that month → reasonable to fill with 0.
- Missing `ticket_count` likely means no support tickets → reasonable to fill with 0.
- Missing `avg_csat` means either no complaints or not available → do not replace them as 0, leave them be.
- Missing `gross_billed` may mean no invoice that month → usually fill with 0 for revenue analysis.

In [32]:
zero_fill_cols = [
    'invoice_count','gross_billed','net_collected','refund_total','paid_invoices',
    'monthly_logins','monthly_reports','monthly_api_calls','avg_active_users','max_storage_gb',
    'ticket_count','high_priority_tickets'
]

customer_month = (
    customer_month
    .assign(**{col: customer_month[col].fillna(0) for col in zero_fill_cols})
)

# Do not fill avg_csat or avg_days_to_pay with zero; zero would be a false business value.
customer_month[['customer_id','month','gross_billed','monthly_logins','ticket_count','avg_csat']].head()

,customer_id,month,gross_billed,monthly_logins,ticket_count,avg_csat
0,C1000,2025-01,2646.0,128.0,0.0,NaN
1,C1000,2025-02,0.0,107.0,0.0,NaN
2,C1000,2025-03,0.0,133.0,1.0,4.0
3,C1000,2025-04,0.0,107.0,0.0,NaN
4,C1000,2025-05,0.0,106.0,0.0,NaN


## 11. Add derived features

Feature construction turns prepared data into managerial evidence.

Examples:

- customer age in months
- support intensity
- usage intensity
- low-engagement flag
- payment risk flag

In [33]:
customer_month = (
    customer_month
    .assign(
        customer_age_months = lambda d: (d['month'].astype(int) - d['signup_month'].astype(int)).clip(lower=0),
        support_per_100_logins = lambda d: np.where(d['monthly_logins'] > 0, 100*d['ticket_count']/d['monthly_logins'], np.nan),
        reports_per_active_user = lambda d: np.where(d['avg_active_users'] > 0, d['monthly_reports']/d['avg_active_users'], np.nan),
        low_engagement = lambda d: (d['monthly_logins'] < 10) & (d['customer_age_months'] >= 2),
        payment_risk = lambda d: (d['pct_paid'].fillna(1) < 0.8) | (d['avg_days_to_pay'].fillna(0) > 15)
    )
)

customer_month[['customer_id','month','customer_age_months','support_per_100_logins','low_engagement','payment_risk']].head(10)

,customer_id,month,customer_age_months,support_per_100_logins,low_engagement,payment_risk
0,C1000,2025-01,6,0.000000,False,False
1,C1000,2025-02,7,0.000000,False,False
2,C1000,2025-03,8,0.751880,False,False
3,C1000,2025-04,9,0.000000,False,False
4,C1000,2025-05,10,0.000000,False,False
5,C1000,2025-06,11,0.000000,False,False
6,C1000,2025-07,12,0.000000,False,False
7,C1000,2025-08,13,1.030928,False,False
8,C1000,2025-09,14,0.000000,False,False
9,C1000,2025-10,15,0.000000,False,False


## 12. Analysis example: customer risk summary

Now that we have a customer-month table, we can create managerial summaries.

In [35]:
risk_summary = (
    customer_month
    .groupby(['month','size_segment'], as_index=False)
    .agg(
        customers = ('customer_id','nunique'),
        avg_logins = ('monthly_logins','mean'),
        pct_low_engagement = ('low_engagement','mean'),
        pct_payment_risk = ('payment_risk','mean'),
        total_net_collected = ('net_collected','sum')
    )
)

risk_summary.tail(10)

,month,size_segment,customers,avg_logins,pct_low_engagement,pct_payment_risk,total_net_collected
14,2025-08,Mid-Market / Enterprise,51,205.431373,0.078431,0.000000,25328.99
15,2025-08,SMB,89,155.707865,0.123596,0.134831,26460.00
16,2025-09,Mid-Market / Enterprise,51,267.803922,0.117647,0.098039,22173.24
17,2025-09,SMB,89,198.337079,0.123596,0.056180,24467.39
18,2025-10,Mid-Market / Enterprise,51,188.313725,0.117647,0.078431,19915.03
19,2025-10,SMB,89,150.247191,0.157303,0.112360,24582.90
20,2025-11,Mid-Market / Enterprise,51,189.215686,0.156863,0.058824,25723.15
21,2025-11,SMB,89,159.123596,0.168539,0.033708,31680.83
22,2025-12,Mid-Market / Enterprise,51,273.274510,0.156863,0.058824,21537.82
23,2025-12,SMB,89,186.617978,0.179775,0.011236,31870.27


In [36]:
# Which customers look risky in the latest month?
latest_month = customer_month['month'].max()

latest_risk = (
    customer_month
    .query('month == @latest_month')
    .assign(risk_score = lambda d: d['low_engagement'].astype(int) + d['payment_risk'].astype(int) + (d['ticket_count'] >= 3).astype(int))
    .sort_values(['risk_score','monthly_logins','ticket_count'], ascending=[False, True, False])
    [['customer_id','industry','region','monthly_logins','ticket_count','avg_csat','net_collected','low_engagement','payment_risk','risk_score']]
)

latest_risk.head(10)

,customer_id,industry,region,monthly_logins,ticket_count,avg_csat,net_collected,low_engagement,payment_risk,risk_score
647,C1053,Retail,South,0.0,1.0,4.0,0.0,True,False,1
719,C1059,Manufacturing,Midwest,0.0,1.0,5.0,0.0,True,False,1
1175,C1097,Healthcare,South,0.0,1.0,4.0,0.0,True,False,1
1199,C1099,Healthcare,West,0.0,1.0,4.0,0.0,True,False,1
1547,C1128,Manufacturing,International,0.0,1.0,3.0,0.0,True,False,1
35,C1002,Healthcare,South,0.0,0.0,NaN,0.0,True,False,1
47,C1003,Retail,West,0.0,0.0,NaN,0.0,True,False,1
203,C1016,Manufacturing,South,0.0,0.0,NaN,0.0,True,False,1
431,C1035,Healthcare,South,0.0,0.0,NaN,0.0,True,False,1
443,C1036,Retail,South,0.0,0.0,NaN,0.0,True,False,1


In [37]:
#Identify the riskiest customers
customer_risk = (
    customer_month
    .assign(risk_score = lambda d: d['low_engagement'].astype(int) + d['payment_risk'].astype(int) + (d['ticket_count'] >= 3).astype(int))
    .groupby("customer_id", as_index=False)
    .agg(
        total_risk_score=("risk_score", "sum"),
        avg_risk_score=("risk_score", "mean"),
        months_observed=("month", "nunique")
    )
)

top_10_risky = customer_risk.nlargest(
    10,
    "total_risk_score"
)

top_10_risky

,customer_id,total_risk_score,avg_risk_score,months_observed
43,C1043,12,1.000000,12
59,C1059,12,1.000000,12
78,C1078,12,1.000000,12
83,C1083,12,1.000000,12
44,C1044,11,0.916667,12
16,C1016,10,0.833333,12
3,C1003,9,0.750000,12
49,C1049,8,0.666667,12
62,C1062,7,0.583333,12
65,C1065,7,0.583333,12


In [39]:
#Identify Top 10 low_engagement customers
customer_LErisk = (
    customer_month
    .assign(le_score = lambda d: d['low_engagement'].astype(int))
    .groupby("customer_id", as_index=False)
    .agg(
        le_score=("le_score", "sum")
    )
)

top_10_lerisky = customer_LErisk.nlargest(
    10,
    "le_score"
)

top_10_lerisky

,customer_id,le_score
43,C1043,12
59,C1059,12
78,C1078,12
83,C1083,12
44,C1044,11
16,C1016,10
3,C1003,9
49,C1049,8
62,C1062,7
92,C1092,7


## 13. Practice exercise

Use the same dataset, but answer a different question:

> Which acquisition channels produce customers with the strongest first-three-month engagement?

Create a table with one row per `acquisition_channel` and the following fields:

1. number of customers
2. average first-three-month logins
3. average first-three-month reports created
4. average first-three-month support tickets
5. percentage of customers with at least one high-priority ticket in the first three months

**Hint:** Use `customer_age_months` and filter to months 0, 1, and 2.

In [ ]:
# PRACTICE EXERCISE STARTER
# Step 1: filter customer_month to the first three months of each customer's life.
# Step 2: aggregate to one row per customer first.
# Step 3: aggregate the customer-level table by acquisition_channel.

first_three_months = customer_month[customer_month['customer_age_months'].between(0, 2)].copy()

# TODO: create customer_first3 table
# customer_first3 = (...)

# TODO: create channel_summary table
# channel_summary = (...)

# display(channel_summary)

## 14. Optional extension

Create a second version of the channel summary where you exclude customers who signed up after October 2025, because they have not had three full months of observation. Explain how this changes the interpretation.

In [ ]:
# OPTIONAL EXTENSION STARTER
# eligible_customers = customers_wr[customers_wr['signup_month'] <= pd.Period('2025-10', freq='M')]
# first_three_eligible = first_three_months[first_three_months['customer_id'].isin(eligible_customers['customer_id'])]
# Continue from there.

## 15. Takeaway

Data wrangling is not just syntax. It is the process of deciding:

- what one row should represent,
- which source tables need to be summarized first,
- what missing values mean after transformations,
- which derived features support the business question, and
- how much the final claims depend on those choices.